# report09 — 챔버 바닥의 함정 — **유령 표적**

**핵심.** 반사되는 챔버 바닥은 **드론을 거쳐 바닥에 튄 경로**를 만들고, 이 경로는 드론과 함께 도플러가 실려 직접파 제거(ECA)를 통과한다 — 진짜 드론 **+3.53 m** 뒤에 **유령 표적**으로 남아, 대역이 넓을수록(5G) **100% 오검출**을 만든다.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | Sionna RT `PathSolver` 는 방의 경로(직접파·바닥반사·표적경유)를 **열거**해 주지만, 그 뒤에 필요한 **레이더 신호처리(직접파·클러터 제거·검출)가 없다** — 정지 클러터와 움직이는 표적을 갈라낼 수단이 스톡 파이프라인에 없다. |
| **② 선행 연구의 방식** | 패시브 레이더는 이 문제를 **표준 체인 ECA→CAF→CFAR** 로 푼다 — 직접파와 그 지연복제로 설명되는 정지 신호 공간을 통째로 사영·제거(Colone et al., *IEEE TAES*). 5G NR 실측(Wypich & Zielinski, *Sensors* 2026, DOI 10.3390/s26041317)·상시-SSB 패시브 5G 드론(Jopanya & Osorio, SPAWC 2025, arXiv:2504.02641)이 같은 체인을 쓴다. |
| **③ 쓴 라이브러리·결합** | 레이더 신호처리(ECA·거리-도플러·CFAR)는 `src/passive_process.py` 가 하고, 그 입력 경로·지연·도플러는 **Sionna RT** 가 준다 — 전파는 Sionna, 신호처리는 우리 코드로 나눠 **중복계산이 없다**. 이 검출 사슬은 표준 **ECA→CAF→CFAR** 그대로이며, 오픈소스 **pyAPRiL**(GPLv3)로 교차검증해 NR/WiFi/LTE 모두에서 표적이 정답 거리빈에 검출됨을 확인했다(`outputs/verify_pyapril.json`). |
| **④ 검증** | 유령 경로의 지연·기하를 **거울상법+프레넬 손계산**으로 독립 대조 — 바닥 반사 여분지연 19.3 ns·세기 -14.7 dB 가 Sionna RT 잔향 탭과 눈금까지 겹치고, 유령이 진짜 뒤 +3.53 m 에 뜬다. |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 바닥 경로 지연·각도·세기 (손계산) | 거울상법 + 프레넬 반사계수 | 닫힌형 |
| 콘크리트 유전율 ε_r = 5.24 | ITU-R P.2040 (3.5 GHz) | 규격 |
| 챔버 잔향 탭 (실측 CIR) | Sionna RT `PathSolver` | 측정 |
| 직접파 제거 ECA | Colone et al., *IEEE TAES* | 문헌 |
| 바이스태틱 기하 (TX/RX 좌표) | src/bistatic_scene.py | 설계값 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sionna-rt` | Sionna RT `PathSolver` — 전파 광선추적. 경로별 **지연 τ · 도플러 f_d · 복소이득 · 반사점 좌표**를 준다 | 🟢 **Sionna 내부** (Mitsuba 3 / OptiX, GPU) |
| `radar-dsp` | 레이더 신호처리 (`src/passive_process.py`) — ECA(직접파 제거) · 거리-도플러 · CA-CFAR | 🔴 **별도** (numpy, CPU). **Sionna 에 레이더 DSP 는 없다** |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `mitsuba` | 3.8.0 | Sionna RT 의 렌더러·광선추적 백엔드 (OptiX, GPU). SBR 도 이걸 쓴다 |
| `torch` | 2.12.1 | Sionna PHY 백엔드 — ⚠ Sionna 2.0 은 TensorFlow 가 아니라 **PyTorch** |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: GPU 1장(자동 선택). 챔버 잔향 탭은 RT 측정 재사용. 검증 스크립트는 수 분.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd benchmark && ~/.venvs/py312/bin/python verify_floor_ghost.py   # → outputs/floor_ghost_verify.json
~/.venvs/py312/bin/python src/make_notebook09.py                   # → report09.ipynb
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/floor_ghost_verify.json` | 바닥 반사 손계산↔RT · 반사세기 스윕 · 파형별 유령 |
| `outputs/figures/report3_f4_floor.png` | 바닥 반사: 거울상+프레넬 ↔ Sionna RT |
| `outputs/figures/report3_f8_ghost.png` | 유령 + 대역폭이 가르는 오검출 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **ECA 는 이 모델에서 이상적으로 정지 반사를 지운다.** 실제 장비는 유한한 동적범위와 바닥 반사의 미세한 도플러 퍼짐 때문에 이만큼 완벽하지 못하다. 그 한계는 이 하네스에 **아직 들어있지 않다** → 실제 정지 클러터가 정말 무해한지는 이 리포트로 단정할 수 없다.
- **유령의 세기는 편파 가정에 걸려 있다.** 바닥 입사각 57° 는 콘크리트 브루스터각 부근이라 수직편파(V)의 반사가 이례적으로 약하다(|Γ|≈0.15). 수평편파(H)라면 유령이 더 세진다. 편파는 측정된 값이 아니라 가정이다.
- **유령의 표적 되비침 밝기(RCS)는 진짜 에코와 같다고 1차 근사했다.** 유령은 시선각이 다르므로 실제 밝기는 다르다.
- **흡수체 벽·천장의 −25 dB 는 실측이 아니라 설계 목표다** — 피라미드 형상의 다중반사로 달성되는 값이다.
- **검출기(CFAR) 자체의 문턱이 제대로 눈금 맞춰져 있는지는 여기서 다루지 않는다** → report10. 이 리포트의 '100% 오검출'은 유령이 **물리적으로 별개 표적으로 분해된다**는 뜻이다.
- **여기의 '5G'는 전대역(98 MHz, NR-PRS/풀점유) 기준이다** — 상시 SSB(7.2 MHz)만 켠 5G 는 유령을 분해하지 못해 이 100% 가 적용되지 않는다(§4 적용 범위 참고, report12 G1/G3 구분).

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| report08 (앞) | 표적 RCS·마이크로도플러 — 유령 세기 근사가 빌려 쓰는 표적 밝기 |
| report10 (다음) | 검출기(CFAR) 자체의 문턱이 눈금 맞춰져 있나 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **반무향 (semi-anechoic)** | 벽·천장은 전파 흡수체지만 **바닥은 반사**하는 챔버. 우리 실험실이 이것 |
| **클러터** | 표적이 아닌 것에서 온 반사(벽·바닥·장비). 정지해 있으면 도플러가 0 이다 |
| **도플러** | 물체가 움직여 생기는 주파수 변화. 정지 반사(도플러 0)와 움직이는 표적을 가르는 축 |
| **유령 (ghost)** | TX→**드론**→바닥→RX 경로의 반사. 드론을 거치므로 도플러가 실려, 클러터와 달리 ECA 를 통과해 가짜 표적으로 뜬다 |
| **ECA (직접파 제거)** | 송신기→수신기 직접파와, 그것의 지연된 복제로 설명되는 모든 정지 반사를 빼는 전처리. 세기를 안 보고 '정지 신호 공간'째로 지운다 |
| **거울상법 (image source)** | 반사면을 거울로 보고 수신기를 면 뒤에 대칭 복제해서 반사 경로를 직선으로 펴 지연·각도를 손계산하는 방법 |
| **프레넬 반사계수 Γ** | 면에 부딪힌 파가 반사되는 비율(복소수). 편파·입사각·유전율에 따라 달라진다 |
| **브루스터각** | 수직편파(V)의 반사가 최소가 되는 입사각. 여기 부근이면 V 유령이 이례적으로 약하다 |
| **바이스태틱 거리 R_b** | TX→표적→RX 경로의 총 길이. 이 값이 표적의 '거리 눈금'이다 |
| **거리분해능 ΔR_b = c/B** | 대역폭 B 가 넓을수록 가까운 두 반사를 **따로** 분해한다 |
| **RCS (되비침 밝기)** | 표적이 얼마나 밝게 되비추는가 [m²]. 밝아야 잡힌다 |
| **CFAR** | 주변 잡음 수준을 보고 검출 문턱을 스스로 정하는 검출기 |

</details>

---


# §1. Sionna 의 공백 — 방은 풀지만 표적을 못 골라낸다

드론의 되비침 밝기(RCS)는 앞 리포트에서 다뤘다(report08). 여기서는 드론이 아니라 **드론을 둘러싼 방** — 특히 바닥 — 이 탐지에 무엇을 하는지 본다. 방 안에서 전파가 어떤 길로 흐르는지는 손으로 열거하기 어렵다. 그 경로 목록을 만들어 주는 것이 **Sionna RT 의 경로 솔버(`PathSolver`)** 다 — 장면의 기하·재질로부터 직접파·반사·회절 경로를 모두 열거한다. 선행 ISAC 디지털트윈 연구도 환경의 결정론적 멀티패스(지연·도플러·각도)를 이렇게 RT 로 얻는다(*Deterministic Modeling of Dynamic ISAC Channels*, EuCAP 2026, arXiv:2603.28736).

그런데 **거기서 끝난다.** `PathSolver` 는 경로 목록을 줄 뿐, 그 목록에서 **무엇이 표적이고 무엇이 지워야 할 클러터인지 가려 주지 않는다** — 직접파를 빼는 처리도, 정지 반사를 지우는 처리도, 검출 문턱을 세우는 처리도 스톡 파이프라인에는 없다(Sionna 에 레이더 DSP 가 없다). 이 리포트의 위협은 정확히 그 공백에서 자란다: 반사하는 바닥이 만드는 여러 경로 중 하나가, 아무 처리 없이는 진짜 드론과 구별되지 않는 **유령 표적**으로 남기 때문이다.

아래는 챔버를 Sionna 로 렌더한 모습이다. 벽과 천장은 뾰족한 피라미드 흡수체로 덮여 전파를 먹지만, **바닥(체커판)은 딱딱한 콘크리트라 전파를 반사**한다. 빨간 점이 송신기(TX), 초록 점이 수신기(RX)이고, 둘 다 같은 벽 위에 떨어져 붙어 있다.

![Semi-anechoic chamber: absorbing walls/ceiling, reflective floor](outputs/renders/rt_05_grazing_floor.png)

이 방을 흔히 '무반사(anechoic) 챔버'라 부르지만, 정확히는 **반무향(semi-anechoic)** 이다 — **바닥만 반사**한다. 이 한 면 때문에 `PathSolver` 가 열거하는 경로가 늘어난다:

- **직접파** — TX 에서 RX 로 곧장 가는 가시선.
- **바닥 반사** — TX 에서 **바닥에 튕겨** RX 로. 조금 늦게 도착한다(정지).
- **표적을 거친 바닥 경로** — TX 에서 드론에 맞고, 다시 **바닥에 튕겨** RX 로. 드론을 거쳐 움직인다.

셋 다 RT 는 똑같이 '경로'로 내놓는다 — 어느 것이 무해하고 어느 것이 위협인지는 RT 가 말해 주지 않는다. §2 는 이 경로들이 실재함을 손계산으로 대조하고, §3 은 패시브 레이더가 이 셋을 어떻게 가르는지, §4 는 우리가 그 처리를 어떤 라이브러리로 얹었는지를 본다.

---
# §2. 증거 — 바닥 경로는 실재하고, 손계산과 맞는다

'RT 가 바닥 경로를 열거한다'는 말을 손계산으로 못박는다. 표준 닫힌형 기법 — **거울상법**(반사면을 거울로 보고 RX 를 바닥 아래로 대칭 복제)과 **프레넬 반사계수** — 으로 각 경로의 지연·각도·세기를 RT 와 독립하게 짚는다.

### 2.1 정지 바닥 반사 — 거울상+프레넬 ↔ Sionna RT

| 항목 | 값 |
|---|---|
| 직접파 경로 | 15.07 m |
| 바닥 경유 경로 | 20.86 m |
| 여분 지연 (늦게 도착) | **19.3 ns** |
| 바닥 입사각 (법선 기준) | 46° — 스침각이 아니라 꽤 비스듬히(≈45°) 때린다 |
| 프레넬 |Γ| (콘크리트 ε_r=5.24) | 0.255 |
| **직접파 대비 세기** | **-14.7 dB** |

이 닫힌형 값을 Sionna RT 가 챔버에서 낸 잔향 탭과 나란히 놓으면 **지연도 세기도 눈금까지 겹친다.** 즉 바닥 반사는 상상이 아니라 RT 가 경로 목록에 넣은 실재하는 경로이며, 그 세기·지연은 교과서 전파 이론과 어긋나지 않는다.

![Floor bounce: image source + Fresnel agrees with Sionna RT](outputs/figures/report3_f4_floor.png)

### 2.2 표적을 거친 바닥 경로 — 유령이 서는 자리

**TX → 드론 → 바닥 → RX** 경로를 보자. 같은 반사 기하를 이번엔 **움직이는 드론을 거쳐** 태우면 이 이차 경로가 된다. 물웅덩이 옆을 걷는 사람의 그림자가 사람을 따라 걷듯, 바닥에 비친 드론의 반사는 드론을 따라 움직인다 — 즉 이 경로에는 **도플러가 실린다**(§3 에서 이 차이가 결정적이 된다). 드론을 챔버 중앙(quiet zone)에 두고 이 경로의 좌표를 거울상법+프레넬(표준 닫힌형)로 짚으면:

| 항목 | 값 |
|---|---|
| 진짜 드론의 바이스태틱 거리 R_b | 22.11 m |
| 유령의 R_b | 25.64 m |
| **진짜 드론 뒤로 떨어진 거리** | **+3.53 m** |
| **진짜 에코 대비 세기** | **-18.1 dB** (진짜 에코보다 크게 어둡지만 진짜 에코 SNR 이 충분해 검출) |
| 바닥 입사각 | 57° |

> **유령은 진짜 드론보다 딱 +3.53 m 뒤에, -18.1 dB 어둡게 뜬다.** 이 위치와 세기는 순전히 방의 기하(TX·RX·바닥·드론 위치)로 정해지며, 어떤 통신 신호를 쓰든 거의 같다. 신호에 따라 달라지는 것은 단 하나 — 이 유령을 진짜와 갈라 볼 수 있느냐다.

![.](outputs/renders/anim/radiomap_scan.gif)

<sub>전파 세기 지도를 바닥→위로 훑는다 — 바닥 반사가 왜 유령을 만드는지 위치를 보여준다.</sub>

---
# §3. 선행 연구의 방식 — 패시브 레이더 표준 체인 ECA→CAF→CFAR

RT 가 가려 주지 않는 '무엇이 클러터이고 무엇이 표적인가'를, 패시브 레이더는 **표준 신호처리 체인**으로 푼다 — 직접파 제거(ECA) → 상호모호함수(CAF, 거리-도플러) → CA-CFAR 검출. 이건 우리가 지어낸 처리가 아니라 문헌의 정석이다: 직접파와 그 지연복제로 설명되는 정지 신호 공간을 통째로 사영·제거하는 ECA(Colone et al., *IEEE TAES*)를, 5G NR 실측(Wypich & Zielinski, *Sensors* 2026, DOI 10.3390/s26041317)과 상시-SSB 패시브 5G 드론 탐지(Jopanya & Osorio, SPAWC 2025, arXiv:2504.02641)가 그대로 쓴다.

이 체인의 핵심은 **ECA 가 세기를 보지 않고 '가만히 있는 신호가 사는 공간' 째로 뺀다**는 것이다. 그래서 §2 의 두 바닥 경로가 정반대로 갈린다.

**정지 바닥 반사(도플러 0)는 그 공간 안에 있어 통째로 지워진다.** 세기를 아무리 키워도 마찬가지다 — 바닥 반사 최강 탭을 없음에서 전력 만 배(+40 dB, 직접파보다 강해질 만큼)까지 키워 신호-대-클러터비(SCR)를 다시 재 보면(기준 ×1 = −26 dB 는 보수적으로 낮춰 잡은 클러터 모델이라 RT 실측 바닥반사 −14.7 dB(§2.1)보다도 약하지만, 이 +40 dB 스윕이 그 실측값까지 전부 덮는다):

| 바닥 반사 최강 탭 | SCR [dB] | 탐지확률 Pd |
|---|---|---|
| 없음 | 33.8801 | 1.00 |
| -26 dB (×1) | 33.8801 | 1.00 |
| -6 dB (×10) | 33.8801 | 1.00 |
| +14 dB (×100) | 33.8801 | 1.00 |

> **SCR 이 전 구간에서 2e-09 dB 밖에 안 움직인다** — 정지한 바닥 반사는 세기와 무관하게 탐지에 **무해**하다. 특정 세기에 운 좋게 그런 게 아니라, ECA 가 정지 신호를 구조적으로 지우기 때문이다.

**반면 표적을 거친 바닥 경로(§2.2)는 드론을 따라 움직여 도플러가 실린다.** ECA 의 '정지 신호 공간' 밖에 있으므로 지워지지 않고 검출기에 그대로 노출된다 — 이것이 유령이 살아남는 이유다. 정지 클러터를 지우는 바로 그 표준 체인이, 도플러가 실린 유령 앞에서는 무력하다.

---
# §4. 우리가 쓴 방식 — 표준 검출 사슬(pyAPRiL 로 교차검증) + Sionna RT(경로), 그리고 대역폭의 대가

이 표준 체인의 신호처리(ECA·거리-도플러·CFAR)는 `src/passive_process.py` 가 하고, 그 입력이 되는 경로·지연·도플러는 **Sionna RT** 가 준다. 역할이 겹치지 않아 **중복계산이 없다**: 전파(경로·지연)는 Sionna, 신호처리(직접파·클러터 제거·검출)는 우리 코드다. 이 검출 사슬이 표준 ECA→CAF→CFAR 그대로임은 오픈소스 **pyAPRiL**(GPLv3)로 교차검증했다 — NR/WiFi/LTE 세 모드 모두에서 표적을 정답 거리빈에 검출함을 확인했다(`outputs/verify_pyapril.json`).

이제 유령을 실제로 신호에 주입해 이 체인을 파형별로 돌린다. 레이더가 두 물체를 따로 볼 수 있는 최소 간격이 **거리분해능 ΔR_b = c / B** 다. 대역폭 B 가 넓을수록 ΔR_b 가 작아져 가까운 둘을 분해한다. 유령은 진짜 뒤 **+3.53 m** 고정이니, **분리 ÷ ΔR_b 가 1 을 넘으면** 유령이 자기만의 자리를 얻어 **가짜 표적**이 된다:

| 파형 | 대역폭 B | 거리분해능 ΔR_b | 분리 ÷ ΔR_b | 유령을 별개 표적으로 검출 |
|---|---|---|---|---|
| **5G NR 100MHz** | 98 MHz | 3.1 m | **1.16×** | **100% — 매 시행 가짜 표적** |
| **WiFi 80MHz** | 76 MHz | 4.0 m | **0.89×** | 0% (진짜와 뭉쳐 못 분해) |
| **LTE 20MHz** | 18 MHz | 16.7 m | **0.21×** | 0% (진짜와 뭉쳐 못 분해) |

> ### 5G 는 진짜 드론과 +3.53 m 뒤 유령을 **둘 다** 검출한다
> 5G 의 거리분해능은 3.1 m 인데 유령은 3.53 m 뒤에 있으니, 분리가 분해능의 **1.16 배** — 여유 있게 넘는다. 그래서 매 시행마다 진짜 드론 하나에 **유령 하나가 가짜 표적으로 더 잡힌다(100%).**

![The floor ghost, and why bandwidth turns it into false alarms](outputs/figures/report3_f8_ghost.png)

그림 왼쪽: 진짜 드론(초록 원)과 유령(파란 X)이 거리-도플러 지도에서 둘 다 살아 있다 — 둘 다 도플러가 실려 ECA 가 지우지 못했기 때문이다. 오른쪽: 유령이 진짜 뒤 3.5 m 에 고정된 채, **대역이 넓은 5G 만 그 3.5 m 를 분해**해 유령을 세로 점선 오른쪽(=별개 표적)으로 밀어낸다.

넓은 대역은 보통 '더 정밀하게 본다'는 장점으로 소개된다. 그런데 바닥 유령 앞에서는 바로 그 정밀함이 **가짜 표적을 하나 더 세우는 약점**이 된다. 좁은 대역(LTE)은 유령을 진짜와 뭉개 안 보이지만(0.21×), 그건 안전해서가 아니라 애초에 위치를 그만큼 못 재기 때문이다.

> ⚠️ 여기서 말하는 '100% 오검출'은 유령이 **물리적으로 별개 표적으로 분해된다**는 뜻이다. 그걸 실제로 검출하는 CFAR 문턱이 제대로 눈금 맞춰져 있는지는 **다음 리포트에서** 따로 본다.

> ⚠️ **적용 범위 — 이 표의 '5G'는 전대역(98 MHz) 기준신호 기준이다** (NR-PRS/풀점유, report12 의 G2·G3 모드; 상시-SSB 조명원은 Jopanya & Osorio, SPAWC 2025 의 패시브-5G-드론 설정과 같다). 5G 가 **상시 신호(SSB 7.2 MHz)만** 켠 경우(report12 의 G1)는 ΔR_b≈41.6 m 라 +3.53 m 유령을 전혀 분해하지 못한다 — 유령 오검출은 0% 가 되지만, 대신 표적 위치도 그만큼 못 잰다(LTE 와 같은 이유). 즉 '유령 문제'는 **5G 를 넓게 쓸수록**(측위 세션·고점유) 나타나는 대가다.

---
# 정리

반사되는 챔버 바닥은 탐지를 **두 갈래**로 건드린다:

| 바닥 경로 | 움직이나 | ECA 가 지우나 | 탐지에 미치는 영향 |
|---|---|---|---|
| 가만히 있는 바닥 반사 | ✗ 정지 | ✅ 세기와 무관하게 지운다 | **무해** |
| 드론 거쳐 바닥에 튄 경로 | ✅ 드론 따라 이동 | ❌ 못 지운다 | **유령 표적**(+3.53 m, -18.1 dB) |

핵심은, 바닥이 아무리 세게 반사해도 **가만히만 있으면** 필터가 지워 무해하고, 정작 문제는 **드론을 거쳐 움직이는** 바닥 경로라는 점이다. 그리고 대역이 넓을수록(5G) 이 유령을 진짜와 별개로 분해해 **100% 가짜 표적**을 만든다 — 광대역의 정밀함이 그대로 오검출로 되돌아온다. 이 유령을 걸러내는 것은 앞으로의 탐지 설계가 반드시 안고 가야 할 문제다.

> **다음 리포트: report10 — 검출기 자체가 눈금이 맞나.** 여기서 유령이 '별개 표적'으로 물리적으로 분해되는 걸 봤다. 그런데 그걸 검출하는 CFAR 의 문턱(명목 오경보율 = 실제 오경보율)이 맞는지는 아직 안 봤다. 그게 어긋나면 이 리포트의 100% 도, 앞으로의 모든 탐지확률도 흔들린다.